# 📘 Agentic Architectures 7: Blackboard Systems

Welcome to the seventh notebook in our series on agentic architectures. Today, we explore the **Blackboard System**, a powerful and highly flexible pattern for coordinating multiple specialist agents. This architecture is modeled on the idea of a group of human experts collaborating around a physical blackboard to solve a complex problem.

Instead of a rigid, predefined sequence of agent handoffs, a Blackboard system features a central, shared data store (the 'blackboard') where agents can read the current state of the problem and write their contributions. A dynamic **Controller** observes the blackboard and decides which specialist agent to activate next based on what is needed to move the solution forward. This allows for an opportunistic and emergent workflow.

To highlight its unique advantages, we will compare it to the **sequential multi-agent system** we built previously. We will present both systems with a complex financial query where the optimal path isn't a simple A → B → C sequence. We will demonstrate how the rigid sequential agent follows a suboptimal path, while the Blackboard system's dynamic Controller activates agents in a more logical, data-driven order, resulting in a more efficient and coherent analysis.

### Definition
A **Blackboard System** is a multi-agent architecture where multiple specialist agents collaborate by reading from and writing to a shared, central data repository called the 'blackboard'. A controller or scheduler dynamically determines which agent should act next based on the evolving state of the solution on the blackboard.

### High-level Workflow

1.  **Shared Memory (The Blackboard):** A central data structure holds the current state of the problem, including the user's request, intermediate findings, and partial solutions.
2.  **Specialist Agents:** A pool of independent agents, each with a specific expertise, continuously monitors the blackboard.
3.  **Controller:** A central 'controller' agent also monitors the blackboard. Its job is to analyze the current state and decide which specialist agent is best equipped to make the next contribution.
4.  **Opportunistic Activation:** The Controller activates the chosen agent. The agent reads the relevant data from the blackboard, performs its task, and writes its findings back to the blackboard.
5.  **Iteration:** The process repeats, with the Controller activating different agents in a dynamic sequence, until it determines that the solution on the blackboard is complete.

### When to Use / Applications
*   **Complex, Ill-Structured Problems:** Ideal for problems where the solution path is not known in advance and requires an emergent, opportunistic strategy (e.g., complex diagnostics, scientific discovery).
*   **Multi-Modal Systems:** A great way to coordinate agents that work with different data types (text, images, code), as they can all post their findings to the shared blackboard.
*   **Dynamic Sense-Making:** Situations that require synthesizing information from many disparate, asynchronous sources.

### Strengths & Weaknesses
*   **Strengths:**
    *   **Flexibility & Adaptability:** The workflow is not hardcoded; it emerges based on the problem, making the system highly adaptive.
    *   **Modularity:** It's very easy to add or remove specialist agents without re-architecting the entire system.
*   **Weaknesses:**
    *   **Controller Complexity:** The intelligence of the entire system depends heavily on the sophistication of the Controller. A naive Controller can lead to inefficient or looping behavior.
    *   **Debugging Challenges:** The non-linear, emergent nature of the workflow can sometimes make it harder to trace and debug than a simple sequential process.

## Phase 0: Foundation & Setup

We'll begin with our standard setup process: installing libraries and configuring API keys for Google, LangSmith, and Tavily.

### Step 0.1: Installing Core Libraries

**What we are going to do:**
We will install our standard suite of libraries for this project series.

In [1]:
# !pip install -q -U langchain-google-genai langchain langgraph rich python-dotenv langchain-tavily

### Step 0.2: Importing Libraries and Setting Up Keys

**What we are going to do:**
We will import the necessary modules and load our API keys from a `.env` file.

**Action Required:** Create a `.env` file in this directory with your keys:
```
GOOGLE_API_KEY="your_google_api_key_here"
LANGCHAIN_API_KEY="your_langsmith_api_key_here"
TAVILY_API_KEY="your_tavily_api_key_here"
```

In [2]:
import os
from typing import List, Annotated, TypedDict, Optional
from dotenv import load_dotenv

# LangChain components
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_tavily import TavilySearch
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.runnables import RunnableConfig
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import create_agent

# LangGraph components
from langgraph.graph import StateGraph, END

# For pretty printing
from rich.console import Console
from rich.markdown import Markdown

from utils import extract_text

# --- API Key and Tracing Setup ---
load_dotenv(override=True)

os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGCHAIN_PROJECT"] = "Agentic Architecture - Blackboard (Google)"

for key in ["GOOGLE_API_KEY", "LANGCHAIN_API_KEY", "TAVILY_API_KEY"]:
    if not os.environ.get(key):
        print(f"{key} not found. Please create a .env file and set it.")

print("Environment variables loaded and tracing is set up.")

Environment variables loaded and tracing is set up.


## Phase 1: The Baseline - A Sequential Multi-Agent System (Corrected)

To understand the blackboard's flexibility, we first need a correctly functioning sequential system. The original version failed because specialists didn't use the output of previous steps. We will correct this by ensuring each agent receives the necessary context from the state.

### Step 1.1: Building the (Corrected) Sequential Team

**What we are going to do:**
We will define specialist agents that explicitly use the outputs of their predecessors, then wire them in a fixed, linear sequence.

In [3]:
console = Console()
llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", temperature=0)
search_tool = TavilySearch(max_results=2, name="web_search")

# State for the sequential agent
class SequentialState(TypedDict):
    user_request: str
    news_report: Optional[str]
    technical_report: Optional[str]
    financial_report: Optional[str]
    final_report: Optional[str]

# --- CORRECTED SPECIALIST NODES FOR SEQUENTIAL AGENT ---
# Each specialist is now a full ReAct agent (via create_agent) so tool calls are actually executed.

news_agent_seq = create_agent(
    llm, [search_tool],
    system_prompt="You are an expert News Analyst. Your specialty is collecting the latest news, articles, and sentiment about a company.\n\nYou have access to a web search tool. Your output MUST be a concise report section, formatted in markdown, focusing only on your area of expertise."
)

tech_agent_seq = create_agent(
    llm, [search_tool],
    system_prompt="You are an expert Technical Analyst with web search tool. You specialize in analyzing stock price charts, trends, and technical indicators.\n\nYou have access to a web search tool. Your output MUST be a concise report section, formatted in markdown, focusing only on your area of expertise."
)

finance_agent_seq = create_agent(
    llm, [search_tool],
    system_prompt="You are an expert Financial Analyst with web search tool. You specialize in interpreting financial statements and performance metrics.\n\nYou have access to a web search tool. Your output MUST be a concise report section, formatted in markdown, focusing only on your area of expertise."
)

def news_analyst_node_seq(state: SequentialState, config: RunnableConfig):
    console.print("--- (Sequential) CALLING NEWS ANALYST ---")
    prompt = f"Find the latest major news about the topic in the user's request and provide a concise summary.\n\nUser Request: {state['user_request']}"
    result = news_agent_seq.invoke({"messages": [HumanMessage(content=prompt)]}, config=config)
    return {"news_report": extract_text(result["messages"][-1].content)}

def technical_analyst_node_seq(state: SequentialState, config: RunnableConfig):
    console.print("--- (Sequential) CALLING TECHNICAL ANALYST ---")
    prompt = f"Based on the following news report, conduct a technical analysis of the company's stock.\n\nNews Report:\n{state['news_report']}"
    result = tech_agent_seq.invoke({"messages": [HumanMessage(content=prompt)]}, config=config)
    return {"technical_report": extract_text(result["messages"][-1].content)}

def financial_analyst_node_seq(state: SequentialState, config: RunnableConfig):
    console.print("--- (Sequential) CALLING FINANCIAL ANALYST ---")
    prompt = f"Based on the following news report, analyze the company's recent financial performance.\n\nNews Report:\n{state['news_report']}"
    result = finance_agent_seq.invoke({"messages": [HumanMessage(content=prompt)]}, config=config)
    return {"financial_report": extract_text(result["messages"][-1].content)}

def report_writer_node_seq(state: SequentialState):
    console.print("--- (Sequential) CALLING REPORT WRITER ---")
    prompt = f"""You are an expert report writer. Your task is to synthesize the information from the News, Technical, and Financial analysts into a single, cohesive report that directly answers the user's original request.

User Request: {state['user_request']}

Here are the reports to combine:
---
News Report: {state['news_report']}
---
Technical Report: {state['technical_report']}
---
Financial Report: {state['financial_report']}
"""
    report = llm.invoke(prompt).content
    return {"final_report": report}

# Build the sequential graph
seq_graph_builder = StateGraph(SequentialState)
seq_graph_builder.add_node("news", news_analyst_node_seq)
seq_graph_builder.add_node("tech", technical_analyst_node_seq)
seq_graph_builder.add_node("finance", financial_analyst_node_seq)
seq_graph_builder.add_node("writer", report_writer_node_seq)

# The rigid, hardcoded sequence
seq_graph_builder.set_entry_point("news")
seq_graph_builder.add_edge("news", "tech")
seq_graph_builder.add_edge("tech", "finance")
seq_graph_builder.add_edge("finance", "writer")
seq_graph_builder.add_edge("writer", END)

sequential_app = seq_graph_builder.compile()
print("Corrected sequential multi-agent system compiled successfully.")

Corrected sequential multi-agent system compiled successfully.


### Step 1.2: Testing the Sequential Agent on a Dynamic Problem

Now that the sequential agent is correctly passing context, let's observe its behavior. It will produce a more coherent report, but its *process* will still be inefficient and fail to follow the conditional logic.

In [4]:
class ToolCallLogger(BaseCallbackHandler):
    """Logs every web_search tool invocation with its query arguments."""

    def on_tool_start(self, serialized, input_str, *, run_id, parent_run_id=None, tags=None, metadata=None, inputs=None, **kwargs):
        tool_name = serialized.get("name", "unknown")
        if tool_name == "web_search":
            console.print(f"[bold cyan]🔍 web_search called:[/bold cyan] {input_str}")

    def on_tool_end(self, output, *, run_id, parent_run_id=None, tags=None, **kwargs):
        pass

In [6]:
dynamic_query = "Find the latest major news about Nvidia. Based on the sentiment of that news, conduct either a technical analysis (if the news is neutral or positive) or a financial analysis of their recent performance (if the news is negative)."

console.print(f"[bold yellow]Testing CORRECTED SEQUENTIAL agent on a dynamic query:[/bold yellow]\n'{dynamic_query}'\n")

# Run the graph
final_seq_output = sequential_app.invoke(
    {"user_request": dynamic_query},
    config={"callbacks": [ToolCallLogger()]}
)

console.print("\n--- [bold red]Final Report from Sequential Agent[/bold red] ---")
console.print(Markdown(extract_text(final_seq_output['final_report'])))

Testing CORRECTED SEQUENTIAL agent on a dynamic query:
'Find the latest major news about Nvidia. Based on the sentiment of that news, conduct either a technical analysis 
(if the news is neutral or positive) or a financial analysis of their recent performance (if the news is 
negative).'

--- (Sequential) CALLING NEWS ANALYST ---

🔍 web_search called: {'query': 'latest major news Nvidia sentiment analysis', 'time_range': 'week', 'topic': 
'news'}

🔍 web_search called: {'time_range': 'week', 'topic': 'news', 'query': 'Nvidia latest news major announcements last
7 days'}

🔍 web_search called: {'query': 'Nvidia GTC 2025 news announcements March 2025', 'topic': 'news', 'time_range': 
'month'}

...

--- (Sequential) CALLING TECHNICAL ANALYST ---

--- (Sequential) CALLING FINANCIAL ANALYST ---

--- (Sequential) CALLING REPORT WRITER ---

--- Final Report from Sequential Agent ---

Nvidia (NVDA) Comprehensive Market Analysis Report                                                                 

Date: March 27, 2026                                                                                               
Subject: Synthesis of News Sentiment and Technical Market Positioning                                              
Overall Sentiment: Constructive / Positive                                                                         

-------------------------------------------------------------------------------------------------------------------

Executive Summary                                                                                                  

Nvidia (NVDA) is currently positioned at a critical technical and fundamental juncture. Following a period of      
market dominance and the announcement of next-generation hardware, news sentiment remains overwhelmingly positive. 
In accordance with this bullish outlook, the following report provides a detailed technical analysis of NVDA’s     
current price action, supported by the fundamental financial drivers that are currently shaping investor behavior. 

-------------------------------------------------------------------------------------------------------------------

I. Latest Major News & Sentiment Analysis                                                                          

The prevailing sentiment for Nvidia is Positive, driven by high-impact developments across its consumer,           
enterprise, and specialized AI sectors:                                                                            

 • GTC 2026 Anticipation: The market is heavily focused on the upcoming GTC 2026 conference. Leading analysts from 
   Truist and Cantor have recently raised price targets, suggesting a potential 80% upside fueled by insatiable    
   demand for AI infrastructure.                                                                                   
 • Blackwell Consumer Launch: The transition to the Blackwell architecture has officially entered the consumer     
   market. HP’s announcement of the Omen Max desktops featuring the GeForce RTX 5090 and 5080 confirms the start of
   a major hardware refresh cycle.                                                                                 
 • Strategic Diversification: Nvidia is successfully expanding into life sciences via its BioNeMo platform.        
   Partnerships with industry giants like Novo Nordisk and Recursion Pharmaceuticals are integrating Nvidia’s DGX  
   systems into global drug discovery pipelines, diversifying revenue beyond traditional compute.                  
 • Market Dominance: Nvidia maintains a formidable 90% share of the data center AI chip market, protected by its   
   CUDA software moat and a shift toward high-margin, integrated "system-level" sales.                             

-------------------------------------------------------------------------------------------------------------------

II. Technical Analysis                                                                                             

Given the positive news sentiment, a technical evaluation is required to determine the optimal entry and exit      
points for the current cycle. NVDA is currently exhibiting a volatility contraction pattern, often referred to as  
"coiling."                                                                                                         

Key Technical Indicators                                                                                           

                                                                                                                   
 Indicator          Status               Analysis                                                                  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
 Price Action       $167.52 – $178.68    Si

**Discussion of the (Corrected) Output:**
The agent now produces a full, logical report. However, the execution trace `News → Technical → Financial` reveals its fundamental flaw. It performed **both** a technical and a financial analysis, completely ignoring the user's conditional request ("either... or..."). This is inefficient and demonstrates the rigidity we aim to solve with the Blackboard architecture.

## Phase 2: The Advanced Approach - A Blackboard System (Corrected)

Now, we'll build the Blackboard system. The key to fixing the original's looping behavior is to craft a much more intelligent prompt for the **Controller**, making it aware of its role as a stateful planner.

### Step 2.1: Defining the Blackboard and the (Corrected) Controller

**What we are going to do:**
1.  **Blackboard State:** Define a `BlackboardState` for shared memory.
2.  **Specialist Agents:** Define the specialist nodes. They will be similar to our previous agents.
3.  **Controller (Corrected):** Create a robust `controller_node` with a prompt that explicitly reasons about completed steps and remaining goals. This is the most critical change.

In [7]:
# The Blackboard State holds all information
class BlackboardState(TypedDict):
    user_request: str
    # The central blackboard where agents post their findings as strings
    blackboard: List[str]
    # List of available agents for the controller to choose from
    available_agents: List[str]
    # The controller's next decision
    next_agent: Optional[str]

# Pydantic model for the Controller's decision
# CORRECTION: Added the list of available agents to the field description to guide the LLM's choice.
class ControllerDecision(BaseModel):
    next_agent: str = Field(description="The name of the next agent to call. Must be one of ['News Analyst', 'Technical Analyst', 'Financial Analyst', 'Report Writer'] or 'FINISH'.")
    reasoning: str = Field(description="A brief reason for choosing the next agent.")

# Reusable factory for creating specialist agents for the blackboard
def create_blackboard_specialist(persona: str, agent_name: str):
    system_prompt = f"""You are an expert specialist agent: a {persona}.
Your task is to contribute to a larger goal by performing your specific function.
Read the initial User Request and the current Blackboard for context.
Use your tools to find the required information.
Your output MUST be a concise markdown report focusing on your area of expertise. Sign the report with your name '{agent_name}'.
"""
    agent = create_agent(llm, [search_tool], system_prompt=system_prompt)

    def specialist_node(state: BlackboardState, config: RunnableConfig):
        console.print(f"--- (Blackboard) AGENT '{agent_name}' is working... ---")
        blackboard_str = "\n---\n".join(state["blackboard"])
        human_msg = f"User Request: {state['user_request']}\n\nBlackboard (previous reports):\n{blackboard_str}"
        result = agent.invoke({"messages": [HumanMessage(content=human_msg)]}, config=config)

        final_content = extract_text(result["messages"][-1].content)

        if not final_content.strip():
            # gemini-2.5-flash-lite returns an empty AIMessage after tool use.
            # Re-invoke the LLM with the full conversation to synthesize a report.
            final_content = extract_text(
                llm.invoke(result["messages"] + [
                    HumanMessage(content="Now write your concise markdown report based on the information above.")
                ]).content
            )

        report = f"\n{final_content}"
        #console.print(report)
        return {"blackboard": state["blackboard"] + [report]}
    return specialist_node

# Create the specialist agent nodes
news_analyst_bb = create_blackboard_specialist("News Analyst", "News Analyst")
technical_analyst_bb = create_blackboard_specialist("Technical Analyst", "Technical Analyst")
financial_analyst_bb = create_blackboard_specialist("Financial Analyst", "Financial Analyst")
report_writer_bb = create_blackboard_specialist("Report Writer who synthesizes a final answer from the blackboard", "Report Writer")

# --- THE CORRECTED, INTELLIGENT CONTROLLER NODE ---
# This is the most important fix. The prompt is now much more sophisticated.
def controller_node(state: BlackboardState):
    console.print("--- CONTROLLER: Analyzing blackboard... ---")

    # Use a structured output LLM to make the decision
    controller_llm = llm.with_structured_output(ControllerDecision)

    blackboard_content = "\n\n".join(state['blackboard'])
    agent_list = state['available_agents']

    # The new prompt is state-aware and goal-oriented.
    prompt = f"""You are the central controller of a multi-agent system. Your job is to analyze the shared blackboard and the original user request to decide which specialist agent should run next.

**Original User Request:**
{state['user_request']}

**Current Blackboard Content:**
---
{blackboard_content if blackboard_content else "The blackboard is currently empty."}
---

**Available Specialist Agents:**
{', '.join(agent_list)}

**Your Task:**
1.  Read the user request and the current blackboard content carefully.
2.  Determine what the *next logical step* is to move closer to a complete answer.
3.  Choose the single best agent to perform that step from the list of available agents.
4.  If the user's request has been fully addressed and a final report has been written, choose 'FINISH'. Do not finish until a "Report Writer" has provided a final, synthesized answer.

Provide your decision in the required format.
"""
    decision_result = controller_llm.invoke(prompt)
    console.print(f"--- CONTROLLER: Decision is to call '{decision_result.next_agent}'. Reason: {decision_result.reasoning} ---")

    # The dictionary returned here updates the 'next_agent' key in the graph's state
    return {"next_agent": decision_result.next_agent}

print("Blackboard components and corrected Controller node defined.")

Blackboard components and corrected Controller node defined.


### Step 2.2: Building the Blackboard Graph

Now we wire the components into a dynamic graph. The Controller acts as a central router. After any specialist runs, control always returns to the Controller to decide the next step.

In [8]:
bb_graph_builder = StateGraph(BlackboardState)

# Add all nodes to the graph
bb_graph_builder.add_node("Controller", controller_node)
bb_graph_builder.add_node("News Analyst", news_analyst_bb)
bb_graph_builder.add_node("Technical Analyst", technical_analyst_bb)
bb_graph_builder.add_node("Financial Analyst", financial_analyst_bb)
bb_graph_builder.add_node("Report Writer", report_writer_bb)

bb_graph_builder.set_entry_point("Controller")

# This function defines the dynamic routing logic based on the Controller's decision
def route_to_agent(state: BlackboardState):
    return state["next_agent"]

# The conditional edges route from the Controller to the chosen specialist or to the end
bb_graph_builder.add_conditional_edges(
    "Controller",
    route_to_agent,
    {
        "News Analyst": "News Analyst",
        "Technical Analyst": "Technical Analyst",
        "Financial Analyst": "Financial Analyst",
        "Report Writer": "Report Writer",
        "FINISH": END
    }
)

# After any specialist runs, control always returns to the Controller for the next decision
bb_graph_builder.add_edge("News Analyst", "Controller")
bb_graph_builder.add_edge("Technical Analyst", "Controller")
bb_graph_builder.add_edge("Financial Analyst", "Controller")
bb_graph_builder.add_edge("Report Writer", "Controller")

blackboard_app = bb_graph_builder.compile()
print("Blackboard system compiled successfully.")

Blackboard system compiled successfully.


## Phase 3: Head-to-Head Comparison

Let's run our new Blackboard system on the same dynamic task and observe its intelligent workflow.

In [9]:
console.print(f"[bold green]Testing BLACKBOARD system on the same dynamic query:[/bold green]\n'{dynamic_query}'\n")

agent_list = ["News Analyst", "Technical Analyst", "Financial Analyst", "Report Writer"]
initial_bb_input = {"user_request": dynamic_query, "blackboard": [], "available_agents": agent_list}

# We use stream to observe the step-by-step process.
# Each chunk from .stream() is {"NodeName": {state_update}}, so we must
# extract the nested state to track the blackboard across iterations.
blackboard_so_far = []
for chunk in blackboard_app.stream(initial_bb_input, {"recursion_limit": 10, "callbacks": [ToolCallLogger()]}):
    for node_name, node_output in chunk.items():
        if "blackboard" in node_output:
            blackboard_so_far = node_output["blackboard"]

    console.print("\n--- [bold purple]Current Blackboard State[/bold purple] ---")
    for i, report in enumerate(blackboard_so_far):
        console.print(f"--- Report {i+1} ---")
        console.print(Markdown(report))
    console.print("\n")

console.print("\n--- [bold green]Final Report from Blackboard System[/bold green] ---")
final_report_content = extract_text(blackboard_so_far[-1])
console.print(Markdown(final_report_content))

Testing BLACKBOARD system on the same dynamic query:
'Find the latest major news about Nvidia. Based on the sentiment of that news, conduct either a technical analysis 
(if the news is neutral or positive) or a financial analysis of their recent performance (if the news is 
negative).'

--- CONTROLLER: Analyzing blackboard... ---

--- CONTROLLER: Decision is to call 'News Analyst'. Reason: The blackboard is empty, and the first step of the user
request is to find the latest major news about Nvidia to determine the sentiment for subsequent analysis. ---

--- Current Blackboard State ---

--- (Blackboard) AGENT 'News Analyst' is working... ---

...

Technical Analysis Report: Nvidia (NVDA) - March 27, 2026                                                          

Overview                                                                                                           

Following the Positive sentiment identified in the latest news—specifically the breakthrough announcement of DLSS 5
and record-shattering Q3 2026 financials—Nvidia's stock (NVDA) continues to exhibit strong bullish characteristics.
The technical indicators reflect a market that is aggressively pricing in the company's sustained dominance in the 
AI and neural rendering sectors.                                                                                   

Key Technical Indicators                                                                                           

 • Price Action & Trend: NVDA is currently trading in a well-defined ascending channel. The stock has recently     
   broken out of a three-month consolidation phase following the Q3 earnings beat, confirming a "bull flag"        
   breakout on the weekly chart.                                                                                   
 • Moving Averages: The stock remains comfortably above its 50-day Moving Average ($132.50) and 200-day Moving     
   Average ($118.20). The "Golden Cross" observed earlier in the year remains intact, signaling long-term upward   
   momentum.                                                                                                       
 • Relative Strength Index (RSI): The RSI currently sits at 74.2, indicating that the stock is in "overbought"     
   territory. While this often suggests a potential short-term cooling period, in high-growth AI cycles, NVDA has  
   historically maintained elevated RSI levels for extended periods during major product launches (like DLSS 5).   
 • MACD (Moving Average Convergence Divergence): The MACD line is trending significantly above the signal line with
   an expanding histogram, confirming strong buying pressure and accelerating momentum.                            

Support and Resistance Levels                                                                                      

 • Immediate Resistance: $145.00 (Psychological barrier and recent all-time high). A sustained break above this    
   level, supported by DLSS 5 adoption news, could open the path toward $160.00.                                   
 • Primary Support: $128.00 (Previous resistance turned support). This level aligns with the 20-day Exponential    
   Moving Average (EMA).                                                                                           
 • Secondary Support: $115.00 (Major structural support). Even in the event of a broader market correction, this   
   level is expected to see significant institutional "buy-the-dip" activity.                                      

Volume Analysis                                                                                                    

Trading volume has seen a 25% increase over the 30-day average during the last three sessions. This "high-volume   
breakout" validates the price move, suggesting that institutional investors are accumulating shares in anticipation
of the DLSS 5 launch this fall.                                                                                    

Conclusion                                                                                                         

The technical outlook for NVDA is Strongly Bullish. Despite overbought RSI readings, the fundamental catalysts     
(DLSS 5 and $57B revenue) provide the necessary "fuel" for the current trend. Traders should look for minor        
pullbacks to the 20-day EMA ($128-$130) as potential entry points, while long-term holders should monitor the $145 
resistance for the next major leg up.                                                                              

Technical Analyst

--- Report 3 ---

Nvidia (NVDA) Comprehensive Market Report - March 27, 2026                                                         

Executive Summary                                                                                                  

Nvidia continues to demonstrate unprecedented market dominance, driven by record-breaking financial performance and
a major technological breakthrough in neural rendering. Following a Positive sentiment assessment of recent news, a
technical analysis confirms a Strongly Bullish outlook for the stock (NVDA), despite short-term overbought         
conditions.                                                                                                        

1. Latest Major News & Sentiment                                                                                   

 • Technological Breakthrough: Nvidia announced DLSS 5, utilizing "Neural Rendering" to achieve Hollywood-level    
   realism in real-time gaming. This is viewed as the most significant graphics advancement in a decade.           
 • Financial Performance: The company reported record Q3 2026 revenue of $57 billion (up 62% YoY), with data center
   revenue alone accounting for $51 billion. Forward guidance for Q4 is set at a robust $65 billion.               
 • Operational Innovation: A new "AI Token" bonus system for engineers aims to 10x productivity by providing       
   compute credits as compensation.                                                                                
 • Sentiment: Positive. The minor "unlaunching" of the 595.59 driver is considered a negligible operational hurdle 
   compared to the massive growth in AI infrastructure and consumer tech.                                          

2. Technical Analysis (Sentiment-Driven)                                                                           

Given the positive news cycle, the technical profile of NVDA reflects aggressive institutional accumulation:       

 • Trend & Price Action: NVDA is in a clear ascending channel and has recently executed a "bull flag" breakout on  
   the weekly chart.                                                                                               
 • Key Indicators:                                                                                                 
    • Moving Averages: The stock is trading well above its 50-day ($132.50) and 200-day ($118.20) averages,        
      maintaining a long-term "Golden Cross."                                                                      
    • Momentum: The RSI stands at 74.2, indicating overbought territory. However, the MACD shows expanding positive
      histograms, suggesting that momentum is still accelerating.                                                  
    • Volume: A 25% increase in trading volume over the 30-day average validates the recent price breakout.        
 • Support & Resistance:                                                                                           
    • Resistance: $145.00 (Psychological/All-time high).                                                           
    • Support: $128.00 (Primary/20-day EMA) and $115.00 (Major structural support).                                

Conclusion                                                                                                         

Nvidia is currently in a "virtuous cycle" where fundamental excellence (DLSS 5 and $57B revenue) is perfectly      
mirrored by bullish technical indicators. While the RSI suggests a potential minor cooling period, the high-volume 
breakout and strong support levels indicate that any pullbacks to the $128–$130 range represent significant buying 
opportunities for the next leg toward $160.                                                                        

Report Writer

--- Final Report from Blackboard System ---

Nvidia (NVDA) Comprehensive Market Report - March 27, 2026                                                         

Executive Summary                                                                                                  

Nvidia continues to demonstrate unprecedented market dominance, driven by record-breaking financial performance and
a major technological breakthrough in neural rendering. Following a Positive sentiment assessment of recent news, a
technical analysis confirms a Strongly Bullish outlook for the stock (NVDA), despite short-term overbought         
conditions.                                                                                                        

1. Latest Major News & Sentiment                                                                                   

 • Technological Breakthrough: Nvidia announced DLSS 5, utilizing "Neural Rendering" to achieve Hollywood-level    
   realism in real-time gaming. This is viewed as the most significant graphics advancement in a decade.           
 • Financial Performance: The company reported record Q3 2026 revenue of $57 billion (up 62% YoY), with data center
   revenue alone accounting for $51 billion. Forward guidance for Q4 is set at a robust $65 billion.               
 • Operational Innovation: A new "AI Token" bonus system for engineers aims to 10x productivity by providing       
   compute credits as compensation.                                                                                
 • Sentiment: Positive. The minor "unlaunching" of the 595.59 driver is considered a negligible operational hurdle 
   compared to the massive growth in AI infrastructure and consumer tech.                                          

2. Technical Analysis (Sentiment-Driven)                                                                           

Given the positive news cycle, the technical profile of NVDA reflects aggressive institutional accumulation:       

 • Trend & Price Action: NVDA is in a clear ascending channel and has recently executed a "bull flag" breakout on  
   the weekly chart.                                                                                               
 • Key Indicators:                                                                                                 
    • Moving Averages: The stock is trading well above its 50-day ($132.50) and 200-day ($118.20) averages,        
      maintaining a long-term "Golden Cross."                                                                      
    • Momentum: The RSI stands at 74.2, indicating overbought territory. However, the MACD shows expanding positive
      histograms, suggesting that momentum is still accelerating.                                                  
    • Volume: A 25% increase in trading volume over the 30-day average validates the recent price breakout.        
 • Support & Resistance:                                                                                           
    • Resistance: $145.00 (Psychological/All-time high).                                                           
    • Support: $128.00 (Primary/20-day EMA) and $115.00 (Major structural support).                                

Conclusion                                                                                                         

Nvidia is currently in a "virtuous cycle" where fundamental excellence (DLSS 5 and $57B revenue) is perfectly      
mirrored by bullish technical indicators. While the RSI suggests a potential minor cooling period, the high-volume 
breakout and strong support levels indicate that any pullbacks to the $128–$130 range represent significant buying 
opportunities for the next leg toward $160.                                                                        

Report Writer

**Discussion of the (Corrected) Output:**
Success! The `GraphRecursionError` is gone. The execution trace reveals a far more intelligent process:

1.  **Controller Start:** The Controller starts and, seeing an empty blackboard, correctly decides to call the **News Analyst** first.
2.  **News Analyst Runs:** The News Analyst finds the latest news and posts its report to the blackboard.
3.  **Controller Re-evaluates:** Control returns to the Controller. It reads the News Analyst's report, understands the sentiment, and follows the user's logic. It intelligently decides to call the appropriate next analyst (**Technical** or **Financial**), completely skipping the other one.
4.  **Specialist Runs:** The chosen analyst performs its task and adds its report to the blackboard.
5.  **Controller Finishes:** The Controller sees all necessary analysis is complete and calls the **Report Writer** to synthesize the final answer.
6.  **Final Call:** After the writer posts the final report, the controller sees this and decides to **FINISH**.

This dynamic, opportunistic workflow is the hallmark of a properly functioning Blackboard system. It perfectly followed the user's complex conditional logic, saving time and resources.

## Phase 4: Quantitative Evaluation

To formalize the comparison, we will use an LLM-as-a-Judge to score both systems on instruction following and process efficiency.

In [10]:
class ProcessLogicEvaluation(BaseModel):
    """Schema for evaluating an agent's logical process."""
    instruction_following_score: int = Field(description="Score 1-10 on how well the agent followed the user's specific conditional instructions (e.g., the 'either/or' logic).")
    process_efficiency_score: int = Field(description="Score 1-10 on whether the agent took the most direct path and avoided unnecessary work.")
    justification: str = Field(description="A brief justification for the scores, referencing specific steps the agent took.")

judge_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0).with_structured_output(ProcessLogicEvaluation)

def evaluate_agent_logic(query: str, final_state: dict):
    # Reconstruct a simplified trace for the judge
    trace = ""
    agent_type = "Unknown"
    if 'blackboard' in final_state: # Blackboard agent
        agent_type = "Blackboard"
        trace = "\n---\n".join(final_state['blackboard'])
    else: # Sequential agent
        agent_type = "Sequential"
        trace = f"1. News Report Generated: {final_state.get('news_report')}\n---\n2. Technical Report Generated: {final_state.get('technical_report')}\n---\n3. Financial Report Generated: {final_state.get('financial_report')}"

    prompt = f"""You are an expert judge of AI agent processes. Your task is to evaluate an agent's performance based on its generated content trace.

**User's Original Task:**
"{query}"

**Agent's Type:** {agent_type}
**Agent's Generated Content Trace:**
```
{trace}
```

**Evaluation Criteria:**
1.  **Instruction Following:** Did the agent respect the conditional logic in the user's task? (e.g., "either a technical analysis... or a financial analysis"). A high score means it followed the logic perfectly. A low score means it ignored it.
2.  **Process Efficiency:** Did the agent avoid doing unnecessary work? A high score means it only ran the required specialists. A low score means it ran specialists that the user's logic explicitly said to skip.

Based on the trace, provide your evaluation.
"""
    return judge_llm.invoke(prompt)

console.print("--- [bold]Evaluating Sequential Agent's Process[/bold] ---")
seq_agent_evaluation = evaluate_agent_logic(dynamic_query, final_seq_output)
console.print(seq_agent_evaluation.dict())

console.print("\n--- [bold]Evaluating Blackboard System's Process[/bold] ---")
bb_agent_evaluation = evaluate_agent_logic(dynamic_query, {"blackboard": blackboard_so_far})
console.print(bb_agent_evaluation.dict())

--- Evaluating Sequential Agent's Process ---

/tmp/ipykernel_116642/159761025.py:41: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  console.print(seq_agent_evaluation.dict())


{
    'instruction_following_score': 1,
    'process_efficiency_score': 1,
    'justification': "The user's instructions clearly stated to conduct EITHER a technical analysis OR a financial 
analysis based on the sentiment of the news. The agent, however, generated a news report, then a technical 
analysis, and then a financial analysis. This means it failed to follow the conditional logic and performed 
unnecessary work by generating both types of analyses when only one was required."
}

--- Evaluating Blackboard System's Process ---

/tmp/ipykernel_116642/159761025.py:45: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  console.print(bb_agent_evaluation.dict())


{
    'instruction_following_score': 10,
    'process_efficiency_score': 10,
    'justification': "The agent correctly identified the sentiment of the news as positive and subsequently 
performed a technical analysis as instructed. It avoided performing a financial analysis, adhering to the 
'either/or' condition. The agent efficiently executed only the necessary steps without performing redundant 
analyses."
}

**Discussion of the Evaluation Output:**
The judge's scores provide a clear, quantitative verdict:

- The **Sequential Agent** will receive a very low `instruction_following_score` (e.g., 2/10) because it blatantly ignored the "either/or" condition. Its `process_efficiency_score` will also be low (e.g., 3/10) because it performed an entire analysis that was explicitly not required.
- The **Blackboard System** will receive near-perfect scores (e.g., 10/10) for both. The judge will recognize that the Controller's dynamic decision-making allowed the system to follow the user's instructions precisely and to operate with maximum efficiency by only activating the necessary specialist.

This evaluation provides definitive proof that for complex, emergent problems where the path forward depends on intermediate results, the flexibility of the Blackboard architecture is vastly superior to a rigid, predefined workflow.

## Conclusion

In this notebook, we have implemented and corrected a **Blackboard System**, demonstrating its significant advantages over a sequential multi-agent architecture. By introducing a shared memory (the blackboard) and an intelligent, state-aware **Controller**, we created a system that is not just collaborative, but also adaptive and opportunistic.

The head-to-head comparison showed that for tasks with conditional logic, the Blackboard system's ability to choose the right expert at the right time leads to a more efficient and logically sound process. While it requires a more sophisticated Controller, this architecture is a powerful tool for tackling the kind of ill-structured, real-world problems that rigid, linear workflows cannot solve effectively.